# Chapter 5: GroupBy & Aggregation

GroupBy is one of Pandas' most powerful features.  
Think of it as the SQL `GROUP BY` on steroids.

1. `groupby` basics — split-apply-combine
2. Single & multiple aggregations with `.agg()`
3. Named aggregations
4. `transform()` vs `apply()` vs `agg()`
5. Filtering groups with `.filter()`
6. Practical example: sales data analysis

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

## 5.1 Sample Sales Data

In [ ]:
n = 200
sales = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=n, freq='D'),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
    'product': np.random.choice(['Widget', 'Gadget', 'Doohickey'], n),
    'quantity': np.random.randint(1, 50, n),
    'unit_price': np.random.choice([9.99, 24.99, 4.99], n),
})
sales['revenue'] = sales['quantity'] * sales['unit_price']
sales.head(10)

## 5.2 GroupBy Basics: Split-Apply-Combine

The GroupBy pattern:
1. **Split** the data into groups based on keys
2. **Apply** a function to each group
3. **Combine** the results into a new DataFrame

In [ ]:
# Group by region
grouped = sales.groupby('region')
print(f'Type: {type(grouped)}')
print(f'Number of groups: {grouped.ngroups}')
print(f'Groups: {list(grouped.groups.keys())}')

In [ ]:
# Basic aggregation
grouped['revenue'].sum()

In [ ]:
# Group by multiple columns
sales.groupby(['region', 'product'])['revenue'].sum()

In [ ]:
# Multiple aggregations at once
sales.groupby('region')['revenue'].agg(['sum', 'mean', 'count', 'std']).round(2)

## 5.3 Advanced Aggregation with `.agg()`

In [ ]:
# Different aggregations per column
result = sales.groupby('region').agg({
    'revenue': ['sum', 'mean'],
    'quantity': ['sum', 'max'],
}).round(2)
result

### Named Aggregations (Pandas ≥ 0.25)

In [ ]:
# Named aggregations — much cleaner column names!
result = sales.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    total_qty=('quantity', 'sum'),
    num_orders=('revenue', 'count'),
).round(2)
result

In [ ]:
# Custom aggregation function
def revenue_range(x):
    return x.max() - x.min()

sales.groupby('region').agg(
    total=('revenue', 'sum'),
    range=('revenue', revenue_range),
    top_10_pct=('revenue', lambda x: x.quantile(0.9)),
).round(2)

## 5.4 `transform()` vs `apply()` vs `agg()`

| Method | Returns | Shape |
|--------|---------|-------|
| `agg()` | Aggregated result | One row per group |
| `transform()` | Same-size Series | Same shape as input |
| `apply()` | Flexible | Depends on the function |

In [ ]:
# transform — result aligns with original DataFrame
# Perfect for adding group-level calculations as new columns

sales['region_avg_revenue'] = sales.groupby('region')['revenue'].transform('mean')
sales['pct_of_region_avg'] = (sales['revenue'] / sales['region_avg_revenue'] * 100).round(1)
sales[['region', 'revenue', 'region_avg_revenue', 'pct_of_region_avg']].head(10)

In [ ]:
# Normalize within each group (z-score)
sales['revenue_zscore'] = sales.groupby('region')['revenue'].transform(
    lambda x: (x - x.mean()) / x.std()
)
sales[['region', 'revenue', 'revenue_zscore']].head()

In [ ]:
# apply — most flexible, but slowest
# Returns the top 2 revenue rows per region
top2 = sales.groupby('region').apply(
    lambda g: g.nlargest(2, 'revenue'),
    include_groups=False
)
top2[['product', 'revenue']].head(8)

## 5.5 Filtering Groups with `.filter()`

In [ ]:
# Keep only regions whose total revenue exceeds a threshold
threshold = sales.groupby('region')['revenue'].transform('sum').median()

high_revenue_regions = sales.groupby('region').filter(
    lambda g: g['revenue'].sum() > threshold
)
print(f'Threshold: {threshold:.2f}')
print(f'Original rows: {len(sales)}')
print(f'Filtered rows: {len(high_revenue_regions)}')
print(f'Remaining regions: {high_revenue_regions["region"].unique()}')

## 5.6 Practical Example: Sales Analysis by Region & Product

In [ ]:
# Revenue by region and product
pivot = sales.groupby(['region', 'product']).agg(
    total_revenue=('revenue', 'sum'),
    avg_order_value=('revenue', 'mean'),
    order_count=('revenue', 'count'),
).round(2)

pivot

In [ ]:
# Top product in each region
top_products = (
    sales.groupby(['region', 'product'])['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue', ascending=False)
    .drop_duplicates(subset='region')
    .sort_values('region')
)
top_products

In [ ]:
# Monthly revenue trend by region
sales['month'] = sales['date'].dt.to_period('M')
monthly = sales.groupby(['month', 'region'])['revenue'].sum().unstack(fill_value=0)
monthly.tail()

In [ ]:
# Clean up helper columns
sales.drop(columns=['region_avg_revenue', 'pct_of_region_avg',
                     'revenue_zscore', 'month'], inplace=True)

## 📝 Key Takeaways

| Concept | Key Point |
|---------|----------|
| **GroupBy** | Split → Apply → Combine |
| **agg()** | Named aggregations produce clean output |
| **transform()** | Returns same-sized output (great for new columns) |
| **apply()** | Most flexible but slowest |
| **filter()** | Keep/discard entire groups based on a condition |